# DermaScope — Segmentación: U-Net con y sin self-attention

Entrena las dos corridas que sostienen §4.2 (segmentación) y la segunda mitad de la
ablación de §4.3: qué aporta un bloque de self-attention (la operación de ViT) en el
cuello de botella de una U-Net con encoder ResNet-34, con los mismos datos, la misma
semilla y el mismo presupuesto de épocas.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

## Esta corrida es larga

A 256px y ~7.000 imágenes de entrenamiento, cada modelo tarda del orden de **40 a 50
minutos** en T4. Aun así, una desconexión a mitad de la segunda corrida costaría la
primera, así que este notebook está diseñado para sobrevivir a una desconexión:

- **Drive se monta al principio**, no al final, y los checkpoints y las métricas se
  escriben directamente ahí. Si la sesión se cae a mitad de la segunda corrida, la
  primera ya está a salvo.
- Las celdas de descarga, extracción y splits son **reanudables**: al reconectar se
  ejecutan de arriba abajo otra vez y saltan lo que ya existe.
- Si prefieres partirlo en dos sesiones, corre hasta la sección 7 en una y retoma
  desde la 8 en otra. El `early_stopping_patience` de 8 épocas puede cortar antes,
  así que los tiempos de arriba son el peor caso.

Esto es lo que lo diferencia de `01_entrenar_clasificador_colab.ipynb`, donde el
guardado a Drive va al final.


## 1. Verificar la GPU

In [ ]:
import torch, sys

if not torch.cuda.is_available():
    raise SystemExit(
        'No hay GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU T4, '
        'y vuelve a ejecutar esta celda.'
    )

print('GPU        :', torch.cuda.get_device_name(0))
print('VRAM       :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('torch      :', torch.__version__)
print('CUDA       :', torch.version.cuda)
print('Python     :', sys.version.split()[0])
print()
print('Anota el nombre de la GPU: va en la tabla de §4.5 del informe.')


## 2. Montar Drive

Va antes del entrenamiento, no después. Todo lo que valga la pena conservar se
escribe directo en Drive desde la primera época.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/dermascope')
(DRIVE_DIR / 'models').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'reports' / 'results').mkdir(parents=True, exist_ok=True)
print('destino:', DRIVE_DIR)


## 3. Traer el código

La misma URL que usaste en el notebook 01.


In [ ]:
REPO_URL = 'https://github.com/USUARIO/dermascope.git'  # <-- reemplazar

import os, subprocess
from pathlib import Path

if 'USUARIO' in REPO_URL:
    raise SystemExit(
        'Falta reemplazar REPO_URL por la URL real del repositorio. '
        'Sin eso el clone falla a mitad de la celda y el error no dice por que.'
    )

REPO_DIR = Path('/content/dermascope')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('\ndirectorio:', Path.cwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


## 4. Dependencias

La U-Net y el bloque de self-attention están escritos con PyTorch y `torchvision`, que
Colab ya trae, así que no hay nada que instalar para entrenar. Igual que en el notebook
01, **no** se instala desde `requirements.txt`: reinstalar torch sobre el de Colab rompe
la compatibilidad con CUDA de la imagen.


In [ ]:
import torch, torchvision, scipy
print('torch      ', torch.__version__)
print('torchvision', torchvision.__version__)
print('scipy      ', scipy.__version__)


In [ ]:
# Verificacion del entorno antes de gastar horas de GPU.
!python scripts/smoke_test.py


## 5. Descargar el dataset

Fuente: Harvard Dataverse, DOI `10.7910/DVN/DBW86T`. Aquí las **máscaras** son tan
necesarias como las imágenes: son la anotación de esta tarea.

Recuerda lo que hay que declarar en el informe: las máscaras de Tschandl se generaron
de forma semiautomática y se revisaron a mano, no son anotación experta píxel a píxel.
Eso acota el Dice alcanzable y conviene decirlo antes de que lo pregunten.


In [ ]:
%%bash
set -e
RAW=/content/data/raw
mkdir -p $RAW

dl() {  # dl <id> <destino>
  if [ -s "$2" ]; then echo "ya existe: $(basename $2)"; return; fi
  curl -sS -L --retry 5 --retry-delay 3 -C - \
    -o "$2" "https://dataverse.harvard.edu/api/access/datafile/$1"
  echo "$(basename $2): $(du -h $2 | cut -f1)"
}

dl 4338392 $RAW/HAM10000_metadata.tab
dl 3838943 $RAW/segmentations.zip
dl 3172585 $RAW/HAM10000_images_part_1.zip
dl 3172584 $RAW/HAM10000_images_part_2.zip

ls -lh $RAW


In [ ]:
# Extraccion reanudable, salta lo ya extraido comparando tamano.
!python scripts/extract_data.py /content/data/raw/HAM10000_images \
    /content/data/raw/HAM10000_images_part_1.zip \
    /content/data/raw/HAM10000_images_part_2.zip

!python scripts/extract_data.py /content/data/raw/HAM10000_segmentations \
    /content/data/raw/segmentations.zip


## 6. Rutas y splits

Diferencia con el notebook 01: `models` y `reports` apuntan a Drive, no a `/content`.
Los checkpoints se escriben ahí en cada época que mejora el Dice, así que una
desconexión cuesta como mucho una época, no el entrenamiento entero.

Los datos sí se quedan en `/content`: son 2,6 GB que el DataLoader lee miles de veces
por época, y leerlos desde Drive montado sería mucho más lento.

La semilla y las proporciones salen de `configs/paths.yaml`, así que el split es
**el mismo** que el del clasificador: sin eso las dos tareas no serían comparables.


In [ ]:
from pathlib import Path

Path('configs/paths.local.yaml').write_text(
    'paths:\n'
    '  ham_metadata: /content/data/raw/HAM10000_metadata.tab\n'
    '  ham_images: /content/data/raw/HAM10000_images\n'
    '  seg_masks: /content/data/raw/HAM10000_segmentations\n'
    '  processed: /content/data/processed\n'
    '  models: /content/drive/MyDrive/dermascope/models\n'
    '  reports: /content/drive/MyDrive/dermascope/reports\n',
    encoding='utf-8',
)
print(Path('configs/paths.local.yaml').read_text())


In [ ]:
!python -m src.data.build_splits --config configs/paths.yaml


## 7. Entrenar — U-Net + self-attention

Encoder ResNet-34 preentrenado en ImageNet, bloque de self-attention sobre el mapa de
8x8 que sale de `layer4` (64 tokens) y decoder U-Net con skip connections. El
checkpoint se elige por `val_dice`.

Pérdida BCE + Dice: la BCE sola sesga hacia el fondo, que domina el área en las
lesiones pequeñas.


In [ ]:
!python -m src.train.train_segmenter --config configs/segmentation.yaml


## 8. Entrenar — U-Net sin self-attention (ablación)

Mismo split, misma semilla, misma pérdida y mismo presupuesto de épocas. El único
factor que cambia es el bloque de self-attention, y eso es lo que permite atribuir la
diferencia en Dice a la atención y no al resto del pipeline.

Si vas a partir el trabajo en dos sesiones, este es el punto de corte: al reconectar,
ejecuta las secciones 1 a 6 otra vez (son rápidas y reanudables) y sigue desde aquí.


In [ ]:
!python -m src.train.train_segmenter --config configs/segmentation.yaml --no-attention


## 9. Comparar U-Net con y sin self-attention

`--skip-localization` porque la métrica de Grad-CAM necesita los dos checkpoints del
**clasificador**, que se entrenan en el notebook 01. Aquí solo se arma la tabla de
segmentación.


In [ ]:
!python -m src.eval.compare_ablation --cls-config configs/classification.yaml --skip-localization


## 10. Verificar lo que quedó en Drive

No hay celda de copia porque no hace falta: todo se escribió en Drive directamente.
Esta celda solo confirma que está.


In [ ]:
from pathlib import Path

for sub in ('models', 'reports/results'):
    carpeta = Path('/content/drive/MyDrive/dermascope') / sub
    print(f'\n{carpeta}:')
    archivos = sorted(p for p in carpeta.glob('*') if p.is_file())
    if not archivos:
        print('   (vacio)')
    for f in archivos:
        print(f'   {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

print('\nPasale reports/results a Claude Code para llenar la model card y el informe.')
